# 전처리 4 · 직접 풀어보는 연습문제 정답지

이 노트북은 `Preprocessing_4_Pipeline_Integration_Practice.ipynb`의 모든 연습문제에 대한 **표준 모범 코드와 결과 해석**을 제공합니다.


In [ ]:
# [실습 환경 준비]
# 이 셀은 파이프라인 실습을 위한 결측치가 포함된 이종 고객 데이터를 생성합니다. 셀을 먼저 실행해주세요.
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

churn_raw_df = pd.DataFrame({
    "age": [25, 45, np.nan, 35, 52, 23, 40, 60, np.nan, 31, 29, 48],
    "spend": [120000, 350000, 280000, np.nan, 510000, 95000, 420000, 680000, 150000, 210000, 180000, 490000],
    "city": ["Seoul", "Busan", "Seoul", np.nan, "Daegu", "Seoul", "Busan", "Seoul", "Incheon", "Seoul", "Daegu", "Busan"],
    "device": ["Mobile", "Web", "Mobile", "Mobile", np.nan, "Mobile", "Web", "Web", "Mobile", "Mobile", "Web", "Mobile"],
    "churn": [0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1]
})


## 문제 1. 수치형과 범주형 피처 리스트 자동 분리 (select_dtypes) [컬럼 분할]

`churn_raw_df`에서 타겟 변수(`churn`)를 분리하고:
1. 독립변수 데이터프레임 `X`와 종속변수 타겟 `y`를 분리하세요.
2. `X.select_dtypes()`를 사용하여 수치형 컬럼 리스트(`num_features`)와 범주형 컬럼 리스트(`cat_features`)를 자동으로 추출하세요.
3. 분리된 각 컬럼 리스트를 출력하세요.


In [ ]:
# 문제 1 풀이
X = churn_raw_df.drop(columns=["churn"])
y = churn_raw_df["churn"]

# 자동 데이터 타입 기반 컬럼 분리
num_features = list(X.select_dtypes(include=["number"]).columns)
cat_features = list(X.select_dtypes(include=["object"]).columns)

print("수치형 피처 목록:", num_features)
print("범주형 피처 목록:", cat_features)


**결과 해석 / 확인한 점:**  
1. **타입 기반 자동 분할:** 하드코딩 대신 `select_dtypes`를 사용하면 향후 데이터프레임에 수십 개의 새로운 컬럼이 추가되어도 수치형(`age`, `spend`)과 범주형(`city`, `device`)이 자동으로 분류되어 파이프라인 유지보수가 간결해집니다.


## 문제 2. 수치형 전용 전처리 파이프라인 구축 (SimpleImputer + StandardScaler) [수치 파이프라인]

수치형 피처를 위한 `num_pipeline`을 작성하세요:
- 1단계: 결측치를 중앙값으로 대체하는 `SimpleImputer(strategy='median')`
- 2단계: 평균 0, 표준편차 1로 표준화하는 `StandardScaler()`
- `num_pipeline.fit_transform(X[num_features])`를 수행하여 결측치가 해결되고 스케일링된 결과를 확인하세요.


In [ ]:
# 문제 2 풀이
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_num_processed = num_pipeline.fit_transform(X[num_features])
print("=== 수치형 파이프라인 처리 결과 (상위 4개) ===")
print(pd.DataFrame(X_num_processed, columns=num_features).head(4).round(3))


**결과 해석 / 확인한 점:**  
1. **체인 연산의 간결성:** 결측 대체와 스케일링이 단 하나의 `Pipeline` 객체 안에서 순차 실행되어 코드 누락의 위험을 원천 제거했습니다.


## 문제 3. 범주형 전용 전처리 파이프라인 구축 (SimpleImputer + OneHotEncoder) [범주 파이프라인]

범주형 피처를 위한 `cat_pipeline`을 작성하세요:
- 1단계: 결측치를 최빈값으로 대체하는 `SimpleImputer(strategy='most_frequent')`
- 2단계: 신규 범주 에러를 방지하는 `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`
- `cat_pipeline.fit_transform(X[cat_features])`를 수행하여 변환된 결과를 확인하세요.


In [ ]:
# 문제 3 풀이
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

X_cat_processed = cat_pipeline.fit_transform(X[cat_features])
cat_colnames = cat_pipeline.named_steps["encoder"].get_feature_names_out(cat_features)

print("=== 범주형 파이프라인 처리 결과 ===")
print(pd.DataFrame(X_cat_processed, columns=cat_colnames).head(4))


**결과 해석 / 확인한 점:**  
1. **문자열 결측의 안전한 처리:** `np.nan` 결측치가 각 컬럼의 최빈값(Seoul, Mobile)으로 먼저 채워진 후 원-핫 인코딩되어 오류 없이 변환되었습니다.


## 문제 4. ColumnTransformer를 통한 이종 데이터 전처리 통합 [전처리 통합]

수치형 파이프라인과 범주형 파이프라인을 하나의 `ColumnTransformer` 객체로 결합하세요:
1. `preprocessor = ColumnTransformer(transformers=[('num', num_pipeline, num_features), ('cat', cat_pipeline, cat_features)])`를 생성하세요.
2. `preprocessor.fit_transform(X)`를 수행하여 최종 변환된 매트릭스의 shape를 출력하세요.


In [ ]:
# 문제 4 풀이
preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

X_all_processed = preprocessor.fit_transform(X)
print(f"원래 입력 X 크기: {X.shape}")
print(f"통합 전처리 후 X 크기: {X_all_processed.shape} (2개 수치 피처 + 6개 원핫 더미 피처)")


**결과 해석 / 확인한 점:**  
1. **이종 데이터의 일괄 병렬 변환:** 수치형 피처 2개와 범주형 피처 2개가 각각의 전용 파이프라인을 거쳐 최종 8차원의 결측 없는 단일 수치 매트릭스로 결합되었습니다.


## 문제 5. 머신러닝 분류기(LogisticRegression)와 결합된 엔드투엔드 Pipeline 완성 [통합 파이프라인]

전처리 객체 `preprocessor`와 최종 모델을 단일 `full_pipeline`으로 결합하세요:
- 분류기: `LogisticRegression(random_state=42)`
1. `Pipeline([('prep', preprocessor), ('model', LogisticRegression(...))])`를 생성하세요.
2. 원본 데이터 `X`, `y`를 전달하여 단 한 번의 호출 `full_pipeline.fit(X, y)`로 학습을 완료하세요.
3. 훈련 데이터에 대한 예측 정확도(`.score(X, y)`)를 소수점 둘째 자리까지 출력하세요.


In [ ]:
# 문제 5 풀이
full_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(random_state=42))
])

# 단 한 번의 호출로 전처리 + 모델 학습 일괄 완료!
full_pipeline.fit(X, y)

train_acc = full_pipeline.score(X, y)
print(f"엔드투엔드 파이프라인 훈련 정확도: {train_acc * 100:.1f}%")


**결과 해석 / 확인한 점:**  
1. **궁극의 추상화:** 결측치 대체, 스케일링, 원-핫 인코딩, 회귀 모델 가중치 학습까지의 방대한 과정이 `full_pipeline.fit(X, y)` 단 한 줄로 완료되었습니다.
2. **누수 없는 완벽한 아키텍처:** 교차 검증(`cross_val_score`) 수행 시에도 폴드 분할마다 전처리가 독립 적용되어 데이터 누수를 100% 차단합니다.


## 문제 6. 실시간 신규 고객 인입 시 단일 객체 추론 (pipeline.predict) [프로덕션 서빙]

웹 서비스 운영 중 결측치가 포함된 신규 고객 데이터가 유입되었습니다:
```python
new_client = pd.DataFrame([{
    "age": np.nan,               # 나이 결측
    "spend": 450000,
    "city": "Incheon",
    "device": "Tablet"            # 학습 데이터에 없던 신규 디바이스
}])
```
1. `full_pipeline.predict(new_client)`와 `full_pipeline.predict_proba(new_client)`를 호출하여 이탈 여부 및 이탈 확률을 예측하세요.
2. 파이프라인이 결측치와 신규 범주를 어떻게 에러 없이 처리했는지 결과를 분석하세요.


In [ ]:
# 문제 6 풀이
new_client = pd.DataFrame([{
    "age": np.nan,
    "spend": 450000,
    "city": "Incheon",
    "device": "Tablet"
}])

# 추론 수행
pred_churn = full_pipeline.predict(new_client)[0]
pred_proba = full_pipeline.predict_proba(new_client)[0][1]

status_str = "이탈 고위험 (Churn=1)" if pred_churn == 1 else "잔류 (Stay=0)"
print(f"신규 고객 예측 결과: {status_str}")
print(f"신규 고객 이탈 확률: {pred_proba * 100:.1f}%")


**결과 해석 / 확인한 점:**  
1. **프로덕션 서빙 내구성:** 나이 결측(`np.nan`)은 학습셋의 중앙값으로 자동 대체되고, 처음 보는 `'Tablet'` 기기는 `handle_unknown='ignore'`에 의해 모든 더미가 0으로 처리되어 시스템 예외 없이 실시간 서빙이 완수되었습니다.
2. **커리큘럼의 피날레:** 이로써 데이터 분석 입문(Stage 1) -> 수치 연산(Stage 2) -> EDA 심화(Stage 3) -> 프로덕션 전처리 파이프라인 구축(Stage 4)으로 이어지는 데이터 분석가/사이언티스트 마스터 커리큘럼이 완성되었습니다.
